In [ ]:
import pyarrow.dataset as ds
import pyarrow.fs as fs
import pyarrow.parquet as pq

In [ ]:
filesystem = fs.HadoopFileSystem("hdfs://arnsdpsbx", port=0)

In [ ]:
remote_path = "/user/team/team_sbertype_evolution/alesha_data/20250905_camp_model/20250919_trx_data/combined_data_for_scoring"

In [ ]:
dataset = ds.dataset(remote_path, filesystem=filesystem, format="parquet")

In [ ]:
from tqdm.autonotebook import tqdm

In [ ]:
import polars as pl

In [ ]:
names = ["date_stamp", "dir_token", "ecom_token", "mcc_token", "brand_token", "city_token", "amt_token"]

In [ ]:
!mkdir /home/datalab/nfs/romashka_data/

In [ ]:
for i, batch in enumerate(tqdm(dataset.to_batches())):
    pl_batch = pl.from_arrow(batch)

    pl_result = (
        pl_batch.with_columns(pl.col("report_dt").dt.timestamp("ms").floordiv(1000))
        .with_columns(pl.col("value").list.eval(pl.element().list.get(j)).alias(name) for j, name in enumerate(names))
        .select("epk_id", "report_dt", *names)
    )

    result_name: str = f"/home/datalab/nfs/romashka_data/batch_{i:04d}.parquet"
    pq.write_table(pl_result.to_arrow(), result_name)

In [ ]:
pl.read_parquet("/home/datalab/nfs/romashka_data/batch_0000.parquet").head(10)